In [ ]:
# ===========================
# Silero VAD + Whisper + SRT
# ===========================
# Requisitos previos:
#   - torch, numpy, openai-whisper, ffmpeg disponible en el sistema.
#   - Silero VAD se descarga vía torch.hub (requiere internet la primera vez).

%pip install -U "numpy<2.0" "numba>=0.61.2" "llvmlite>=0.44,<0.45" "torchaudio" --quiet

import os, math, numpy as np, torch, whisper
from difflib import SequenceMatcher

# ---- Localización de carpetas ----
CARPETA_CODIGO = os.getcwd()
CARPETA_AUDIO  = os.path.join(CARPETA_CODIGO, "audio")
CARPETA_SRT    = os.path.join(CARPETA_CODIGO, "srts")
os.makedirs(CARPETA_SRT, exist_ok=True)

# Tomar el primer WAV que exista en ./audio (si hay varios, coge el más reciente)
wav_files = [f for f in os.listdir(CARPETA_AUDIO) if f.lower().endswith(".wav")]
if not wav_files:
    raise FileNotFoundError("No se encontró ningún archivo .wav en la carpeta './audio'.")
wav_files.sort(key=lambda f: os.path.getmtime(os.path.join(CARPETA_AUDIO, f)), reverse=True)
AUDIO_INPUT = os.path.join(CARPETA_AUDIO, wav_files[0])

# Salida en ./srt/es_subs.srt
SRT_OUTPUT = os.path.join(CARPETA_SRT, "sub_es.srt")
device = "cpu"
print(device)
model = whisper.load_model("medium", device=device)  # cambia a "medium" si te falta VRAM

# -------- utilidades --------
def similar(a, b, th=0.82):
    return SequenceMatcher(None, a.strip(), b.strip()).ratio() >= th

def write_srt(segments, path):
    def fmt(t):
        t = max(0.0, float(t))
        ms = int(round((t - int(t)) * 1000))
        s = int(t) % 60
        m = (int(t)//60) % 60
        h = int(t)//3600
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    with open(path, "w", encoding="utf-8") as f:
        for i, s in enumerate(segments, 1):
            f.write(f"{i}\n{fmt(s['start'])} --> {fmt(s['end'])}\n{s['text'].strip()}\n\n")

def merge_small_gaps(segs, max_gap=0.25):
    if not segs: return segs
    out = [[segs[0][0], segs[0][1]]]
    for s, e in segs[1:]:
        if s - out[-1][1] <= max_gap:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(s, e) for s, e in out]

# -------- VAD con Silero (torch.hub) --------
def vad_silero(audio_f32, sr=16000, threshold=0.5, min_speech=0.30, min_silence=0.20, pad=0.25):
    """
    Devuelve lista de (start_sec, end_sec).
    threshold: 0-1 (más alto => más estricto)
    min_speech/min_silence/pad en segundos
    """
    model_vad, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad', trust_repo=True, verbose=False)
    (get_speech_timestamps, _, _, _, _) = utils

    wav_t = torch.from_numpy(audio_f32)  # float32 [-1,1]
    if wav_t.dim() > 1:
        wav_t = wav_t.mean(dim=0)

    ts = get_speech_timestamps(
        wav_t, model_vad, sampling_rate=sr,
        threshold=threshold,
        min_speech_duration_ms=int(min_speech*1000),
        min_silence_duration_ms=int(min_silence*1000),
        max_speech_duration_s=600
    )
    total = len(audio_f32)/sr
    segs = []
    for t in ts:
        s = max(0.0, t['start']/sr - pad)
        e = min(total, t['end']/sr + pad)
        segs.append((s, e))
    segs = merge_small_gaps(segs, max_gap=0.25)
    return segs

# -------- pipeline principal --------
print(f"Usando audio: {AUDIO_INPUT}")
audio = whisper.load_audio(AUDIO_INPUT)  # float32, 16000 Hz
sr = 16000
total_dur = len(audio)/sr

# 2) Detectar tramos de voz con Silero
if device == "cuda":
    segments = vad_silero(audio, sr=sr, threshold=0.5, min_speech=0.30, min_silence=0.20, pad=0.25)
    print(f"VAD Silero → {len(segments)} segmentos")
else:
    segments = [(0, total_dur)]
# 3) Parámetros de Whisper estables
kw = dict(
    language="es",
    fp16=(device=="cuda"),
    condition_on_previous_text=False,
    temperature=[0.0, 0.2, 0.5],
    beam_size=5, patience=1.0,
    compression_ratio_threshold=2.4,
    logprob_threshold=-0.6,
    no_speech_threshold=0.6,
    verbose=False
)

# 4) Transcribir cada tramo y ajustar tiempos absolutos
all_segments = []
for i, (s, e) in enumerate(segments, 1):
    clip = audio[int(s*sr):int(e*sr)]
    if len(clip) == 0: 
        continue
    print(f"[{i:>3}/{len(segments)}] {100*i/len(segments):5.1f}%  seg {s:7.2f}-{e:7.2f}s", end="\r", flush=True)
    result = model.transcribe(clip, **kw)
    for seg in result["segments"]:
        all_segments.append({
            "start": s + float(seg["start"]),
            "end":   s + float(seg["end"]),
            "text":  seg["text"].strip(),
            "avg_logprob": seg.get("avg_logprob", -10.0),
            "compression_ratio": seg.get("compression_ratio", 0.0),
        })

# 5) Stitch: ordenar, fusionar solapes y deduplicar ecos cortos
all_segments.sort(key=lambda x: (x["start"], x["end"]))
stitched = []
for seg in all_segments:
    if not stitched:
        stitched.append(seg); 
        continue
    last = stitched[-1]
    overlap = min(last["end"], seg["end"]) - max(last["start"], seg["start"])

    # Si hay solape y el texto es esencialmente el mismo → extendemos
    if overlap > 0 and (seg["text"] == last["text"] or similar(seg["text"], last["text"])):
        last["end"] = max(last["end"], seg["end"])
        if seg["avg_logprob"] > last["avg_logprob"]:
            last["text"] = seg["text"]
            last["avg_logprob"] = seg["avg_logprob"]
        continue

    # Eco exacto muy corto típico en bordes de VAD
    if seg["text"] == last["text"] and (seg["end"]-seg["start"]) <= 1.1:
        last["end"] = seg["end"]
        continue

    stitched.append(seg)

# 6) Exportar SRT a ./srt/es_subs.srt
write_srt(stitched, SRT_OUTPUT)
print(f"\nSRT escrito con {len(stitched)} líneas → {SRT_OUTPUT}")

Note: you may need to restart the kernel to use updated packages.
cpu
Usando audio: /home/aramos-m/Escritorio/TFM-Telefonica/TFM-Telefonica/code/audio/video.wav


NameError: name 'segments' is not defined